# Track

Here we demonstrate the impacts of some of the {func}`tams.track` options.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np

import tams

We load a pre-identified dataset so we can skip to the tracking stage.

In [ ]:
ce = tams.data.load_example("mpas-regridded-identify")
ce

In [ ]:
def geoax(*, proj=None):
    if proj is None:
        proj = ccrs.PlateCarree()
    _, ax = plt.subplots(figsize=(8, 5), subplot_kw=dict(projection=proj), layout="constrained")
    ax.gridlines(draw_labels=True)
    ax.add_feature(cfeature.LAND)
    return ax

ce.plot(ec="C0", fc="none", alpha=0.1, ax=geoax(), transform=ccrs.PlateCarree())

In [ ]:
times, ces = zip(*[(time, ce) for time, ce in ce.groupby("time")])
print(len(times))

In [ ]:
%%time

ce = tams.track(ces, times)

In [ ]:
ce.plot(fc="none", alpha=0.1, column="mcs_id", cmap="tab10", ax=geoax(), transform=ccrs.PlateCarree())

In [ ]:
gb = ce.groupby("mcs_id")
(gb.size() / gb.time.nunique()).value_counts().sort_index(ascending=False)

In [ ]:
import warnings

ax = geoax()

colors = plt.get_cmap("tab10").colors

n = 0
for i, (_, g) in enumerate(ce.groupby("mcs_id")):
    # c = colors[i % len(colors)]
    c = colors[n % len(colors)]
    nt = g.time.nunique()
    if nt < 10:
        continue  # more interesting
    if nt == 1:
        alpha = [1]
    else:
        alpha = np.linspace(0.1, 1.0, nt)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # g.centroid.plot(color=c, ax=ax, marker=".", ec="none")
        track = g.dissolve("time").centroid

    for j, (a, b) in enumerate(zip(track.iloc[:-1], track.iloc[1:])):
        ax.plot([a.x, b.x], [a.y, b.y], ls="-", c=c, alpha=alpha[j + 1])

    n += 1
    if n > 20:
        break